# Applied ML Teaching Copilot RAG Starter

A minimal RAG baseline for answering questions from Applied Machine Learning course materials, following the same structure we used in the module: a `search` function, a `build_prompt` function, an `llm` function, and a top-level `rag` function that chains them together.

This notebook loads a small prototype course-material dataset and retrieves relevant records before asking the model to answer.

In [10]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index

openai_client = OpenAI()

## 1. Load Your Data

Load the prototype Applied Machine Learning course-material dataset.

In [11]:
with open("../data/course_materials.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

documents[:2]

[{'id': 'aml-001',
  'module': 'Regression',
  'lesson': 'Model Evaluation',
  'topic': 'MAE vs MSE',
  'content': 'Mean Absolute Error (MAE) averages the absolute size of prediction errors, so each error contributes linearly. Mean Squared Error (MSE) averages squared errors, so large mistakes receive much more weight. Use MAE when you want an evaluation metric that is easy to interpret in the target units and less sensitive to outliers. Use MSE when large errors are especially costly or when you want the training and evaluation process to strongly penalize big misses.',
  'source_type': 'lecture_note'},
 {'id': 'aml-002',
  'module': 'Supervised Learning',
  'lesson': 'Tree-Based Models',
  'topic': 'Decision trees',
  'content': 'A decision tree makes predictions by repeatedly splitting the data into smaller groups using feature-based rules. At each internal node, the tree chooses a split that makes the resulting groups more pure with respect to the target. Decision trees are useful 

## 2. Build the Index

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [12]:
index = Index(
    text_fields=["topic", "content"],
    keyword_fields=["module", "lesson", "source_type"],
)

index.fit(documents)

## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [13]:
def search(query):
    return index.search(query, num_results=5)

In [14]:
instructions = """
You are Applied ML Teaching Copilot, an assistant for instructors and students
working with Applied Machine Learning course materials. Answer the QUESTION
using the retrieved course CONTEXT. Explain concepts clearly and make the
answer useful for teaching, studying, or reviewing the topic. Do not invent
information that is not supported by the context. If the available course
context is insufficient, say so clearly.
""".strip()


def build_prompt(query, search_results):
    context = ""

    for doc in search_results:
        context += f"""
Topic: {doc['topic']}
Module: {doc['module']}
Lesson: {doc['lesson']}
Content: {doc['content']}
""".strip() + "\n\n"

    user_prompt = f"""
<QUESTION>
{query}
</QUESTION>

<CONTEXT>
{context.strip()}
</CONTEXT>
""".strip()

    return user_prompt

In [15]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [16]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

In [17]:
print(rag("When should I use MAE instead of MSE in a regression problem?"))

You should use Mean Absolute Error (MAE) instead of Mean Squared Error (MSE) in a regression problem when you prefer an evaluation metric that is easy to interpret in the target units and when you want to be less sensitive to outliers. 

MAE calculates the average of the absolute errors, treating each error linearly, which makes it straightforward to understand the average magnitude of errors in the same units as your target variable. This is particularly useful when clarity in communication of the model's performance is essential.

In contrast, MSE squares the errors, which means that larger errors disproportionately affect the overall score. Therefore, MSE is more appropriate when large errors are particularly costly, or when you want to penalize those large misses more heavily during model training and evaluation.

In summary, choose MAE for interpretability and robustness against outliers, and choose MSE if you need to heavily penalize large errors.
